In [0]:
%run ../config/set_up_env_paths

In [0]:
from config.core import config
from pyspark.sql.functions import col , when, count, to_date, round, upper, trim, current_date
import pyspark
from pyspark.sql.types import DecimalType

In [0]:
x = spark.table("nbp.bronze.table_a")
display(x.head(4))

In [0]:
table = dbutils.widgets.get("table")

In [0]:
brozne_table = spark.table(f"{config.catalog.catalog_name}.{config.catalog.source_schema}.{table}")

In [0]:
def transform_table(*,bronze_table:pyspark.sql.connect.dataframe.DataFrame ) -> pyspark.sql.connect.dataframe.DataFrame:

    transformed_table = bronze_table.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))\
    .withColumnRenamed("code", "currency_code")\
        .withColumn("price_in_PLN_raw",  col("price").cast(DecimalType(10,9)))\
            .withColumn("price_in_PLN", round(col("price"), 2))\
                .withColumn("currency_code", upper(trim(col("currency_code"))))\
                    .withColumn("ingest_date", current_date())

  
    valid_transformed_table = transformed_table.filter(
        (col("date").isNotNull()) &
        (col("currency_code").isNotNull()) &
        (col("price_in_pln_raw").isNotNull()) &
        (col("price_in_pln_raw") > 0)
    )

    return valid_transformed_table

In [0]:
z = transform_table(bronze_table=x)
display(z)

In [0]:
config.nbp.table_a = transform_table(bronze_table=brozne_table)

In [0]:
table.write.format("delta").mode("overwrite").saveAsTable(f"{config.catalog.catalog_name}.{config.catalog.silver_schema}.{table}")